# Loans at Risk: Capturing Default – EXTRACT-TRANSFORM-LOAD

## Purpose
This notebook implements the **ETL workflow** for LendingClub historical loan data, using 2007–2014 as training data and 2015 as testing data. The goal is to convert raw CSV files into workable datasets suitable for downstream analysis, with a focus on inspection, cleaning, and type validation.

## Packages and Environment
Before running this notebook, it is recommended to set up a **virtual environment** (e.g., `venv` or `conda`) to manage dependencies and ensure reproducibility.

All packages required for this project — including those used in later notebooks for EDA, feature engineering, and modeling — should be installed in this environment. This is done once in the next code block using the project’s `requirements.txt` file.

## Objectives
In this notebook, I will:

1. **Extract** raw loan data from CSV files.
2. **Inspect** the raw datasets to identify missing values, mixed types, constant columns, and other data quality issues.
3. **Transform** the data:
   - Handle null values, type conversions, and columns stored as objects
   - Drop irrelevant or fully-null columns
   - Log key findings in a structured and reproducible way
4. **Load** the cleaned datasets into Parquet files for downstream EDA and modeling.

> Note: This notebook focuses on **data preparation and quality assessment**. EDA, feature engineering, and model training are performed in later notebooks.


> ⚠️ Note: Make sure the correct virtual environment is active before running the cell below. Installing the packages below for the first time may take several minutes. It is recommended to restart the kernel afterwards to ensure all packages load correctly.

In [1]:
# Install required libraries in virtual environment
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

# -------------------------------
# Dynamically find project root by locating the 'data' folder
# -------------------------------
project_root = Path.cwd()
for parent in [project_root] + list(project_root.parents):
    if (parent / "data").exists():
        project_root = parent
        break

# Add src folder to sys.path so we can import custom Python modules
sys.path.append(str(project_root / "src"))

In [3]:
# -------------------------------
# Imports: libraries and custom functions
# -------------------------------

from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from tqdm.notebook import tqdm

# Custom functions from Python scripts package
from python.logging_utils import (
    log_messages,
    get_logger, 
    log_category_differences
)
from python.preprocessing_utils import (
    initial_inspection,
    create_row_identifier,
    apply_credit_sentinel_encoding,
    drop_columns_with_logging,
    audit_string_columns,
    compare_categorical_column_values,
    convert_month_year_columns_to_datetime,
    transform_emp_length,
    normalize_home_ownership,
    normalize_text_columns,
    apply_categorical_mapping
)

In [4]:
# ===============================
# Paths and run context
# ===============================

# Base logs folder
logs_root = project_root / "logs"
logs_root.mkdir(exist_ok=True)

# Log file
project_log_file = logs_root / "project.log"
project_log_file.parent.mkdir(parents=True, exist_ok=True)
project_log_file.touch(exist_ok=True)  # create file if it doesn't exist

# Add a clear visual separator for this run in all logs
separator = "\n" + "="*60 + "\n"
run_header = f"NEW RUN: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}"

for log in [project_log_file]:
    log_messages(separator, log)
    log_messages(run_header, log)
    log_messages("="*60, log)

# Print immediately in the notebook which run is starting
print(run_header)

# Raw data storage path
raw_training_data_file = project_root / "data" / "external" / "loan_data_2007_2014.csv"
raw_training_data_file.parent.mkdir(parents=True, exist_ok=True)

raw_test_data_file = project_root / "data" / "external" / "loan_data_2015.csv"
raw_test_data_file.parent.mkdir(parents=True, exist_ok=True)

# Interim data storage path
clean_training_data_file = project_root / "data" / "interim" / "clean_loan_data_2007_2014.parquet"
clean_training_data_file.parent.mkdir(parents=True, exist_ok=True)

clean_test_data_file = project_root / "data" / "interim" / "clean_loan_data_2015.parquet"
clean_test_data_file.parent.mkdir(parents=True, exist_ok=True)

feature_base_training_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2007_2014.parquet"
feature_base_training_data_file.parent.mkdir(parents=True, exist_ok=True)

feature_base_test_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2015.parquet"
feature_base_test_data_file.parent.mkdir(parents=True, exist_ok=True)


NEW RUN: 2026-02-18 21:22:15


## Initial Data Inspection

This section evaluates the structure, quality, and consistency of the raw LendingClub dataset. The objective is to identify missing values, mixed data types, formatting issues, structural artifacts, and potential anomalies that must be addressed before further analysis.

The inspection focuses on understanding the dataset as delivered, without modification. Structural artifacts (such as exported index columns), identifier fields, and non-informative variables will be documented here and handled explicitly during the transformation phase.


In [5]:
# Loading the csv file into pandas dataframes
df_raw_training_data = pd.read_csv(raw_training_data_file)
df_raw_training_data.head(5)

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_30888\706002346.py:2: DtypeWarning: Columns (0: desc) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw_training_data = pd.read_csv(raw_training_data_file)


,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Initial inspection of the raw training data
inspection_summary_train = initial_inspection(df_raw_training_data, project_log_file)

In [7]:
# Show dataset structure and memory usage of the raw training data
rows, cols = inspection_summary_train["shape"]
memory_mb = inspection_summary_train["memory_usage_mb"]

pd.DataFrame({
    "Rows": [rows],
    "Columns": [cols],
    "Memory Usage (MB)": [round(memory_mb, 2)]
})


,Rows,Columns,Memory Usage (MB)
0,466285,75,389.2


In [8]:
# Show feature-level summary of the raw training data, sorted by percentage of missing values
with pd.option_context("display.max_rows", None,
                       "display.max_columns", None):
    
    display(
        inspection_summary_train["feature_summary"]
        .sort_values("null_percent", ascending=False)
    )

,dtype,n_unique,null_percent,is_numeric,is_object,is_mixed_type,is_numeric_object,is_fully_null,is_constant
max_bal_bc,float64,1,100.00,True,False,False,False,True,False
open_rv_24m,float64,1,100.00,True,False,False,False,True,False
inq_fi,float64,1,100.00,True,False,False,False,True,False
open_rv_12m,float64,1,100.00,True,False,False,False,True,False
il_util,float64,1,100.00,True,False,False,False,True,False
mths_since_rcnt_il,float64,1,100.00,True,False,False,False,True,False
total_bal_il,float64,1,100.00,True,False,False,False,True,False
open_il_24m,float64,1,100.00,True,False,False,False,True,False
open_il_12m,float64,1,100.00,True,False,False,False,True,False
open_il_6m,float64,1,100.00,True,False,False,False,True,False


In [9]:
# Access the feature summary from the inspection
feature_summary_train_df = inspection_summary_train["feature_summary"]

# Build a list of structural issues
structural_issues_records = []

for column_name in feature_summary_train_df.index:
    if feature_summary_train_df.loc[column_name, "is_fully_null"]:
        structural_issues_records.append({"Column": column_name, "Issue": "100% Null"})
    elif feature_summary_train_df.loc[column_name, "null_percent"] > 50:
        structural_issues_records.append({"Column": column_name, "Issue": "High Missing (>50%)"})
    elif feature_summary_train_df.loc[column_name, "is_constant"]:
        structural_issues_records.append({"Column": column_name, "Issue": "Constant"})
    elif feature_summary_train_df.loc[column_name, "is_mixed_type"]:
        structural_issues_records.append({"Column": column_name, "Issue": "Mixed Type"})
    elif feature_summary_train_df.loc[column_name, "is_numeric_object"]:
        structural_issues_records.append({"Column": column_name, "Issue": "Numeric Stored as Object"})

# Convert to DataFrame
structural_issues_train_df = pd.DataFrame(structural_issues_records)

# Define severity order
severity_order = [
    "100% Null",
    "High Missing (>50%)",
    "Constant",
    "Mixed Type",
    "Numeric Stored as Object"
]

# Sort by severity and then by column name
structural_issues_train_df["Issue"] = pd.Categorical(
    structural_issues_train_df["Issue"],
    categories=severity_order,
    ordered=True
)
structural_issues_train_df = structural_issues_train_df.sort_values(
    ["Issue", "Column"]
).reset_index(drop=True)

# Display the sorted structural issues
display(structural_issues_train_df)


,Column,Issue
0,all_util,100% Null
1,annual_inc_joint,100% Null
2,dti_joint,100% Null
3,il_util,100% Null
4,inq_fi,100% Null
5,inq_last_12m,100% Null
6,max_bal_bc,100% Null
7,mths_since_rcnt_il,100% Null
8,open_acc_6m,100% Null
9,open_il_12m,100% Null


## Initial Data Inspection – Training Set (2007–2014)

The raw training dataset contains:

- **Rows:** 466,285  
- **Columns:** 75  
- **Memory usage:** ~389 MB  
- **Data types:** 49 numeric, 26 object  
- **Fully null columns:** 17  
- **Constant columns:** 2 (`policy_code`, `application_type`)  
- **Columns with more than 50% null values:** 4  

### Key Findings

- Seventeen columns contain no observations, and two additional columns (`policy_code`, `application_type`) hold the same value for every record. Since these variables do not contribute any information, they will be removed during ETL rather than corrected or transformed.

- Four columns contain more than 50% null values. Three of these (`mths_since_last_delinq`, `mths_since_last_major_derog`, `mths_since_last_record`) represent credit timing variables where null indicates absence of a prior event. These will be retained and transformed using an explicit indicator variable and a sentinel value (9999) to preserve structural meaning. The remaining high-null column (`desc`) is an optional free-text field and is excluded from this project to maintain a structured modeling scope.

- Aside from the two constant columns noted above, no mixed-type inconsistencies were detected. Removing structurally redundant variables simplifies the dataset without requiring additional type correction.

- At approximately 389 MB, the dataset remains manageable on a standard workstation. However, transformations will be implemented with attention to memory efficiency to avoid unnecessary duplication during feature engineering.

- Finally, the dataset includes payment and recovery-related variables that reflect information only available after loan issuance. These columns will be explicitly flagged during ETL to prevent unintended data leakage under the application-submission prediction definition.

---

After inspection, redundant columns will be removed, the dataset standardized, and the cleaned version stored as a Parquet file for exploratory analysis in Notebook 2.


## Feature Eligibility – Prediction at Application Submission

As described in the project README, the prediction point is defined at loan application submission, before any underwriting, pricing, funding, or servicing decisions take place.

Using the official LendingClub data dictionary, each variable was classified according to when it becomes available in the loan lifecycle. This ensures that all retained features are genuinely available at the time the application is submitted.

---

### Identifier Handling

At the beginning of the transformation phase, a stable internal identifier (`row_id`) is created for each record. This identifier is independent of platform-specific fields and is used solely to ensure consistent alignment across modeling and validation datasets.

Platform identifiers (`id`, `member_id`) and structural artifacts such as `Unnamed: 0` are removed during ETL, as they provide no predictive value and are not required for analysis.

---

### Variables Retained for Modeling

The following information is retained for downstream analysis:

- Borrower-provided application data (income, employment details, requested loan amount, purpose, home ownership, etc.)
- Credit bureau variables reflecting the borrower’s credit state at the time of application

Credit bureau variables available at submission include FICO ranges, delinquency history, revolving utilization, open accounts, inquiry counts, and related credit metrics.

Variables reflecting subsequent credit updates during the loan lifecycle (e.g., later FICO pulls or servicing updates) are excluded to prevent temporal leakage.

Together, these retained variables represent the information set available at the time of application submission.

---

### Variables Excluded from the Training Feature Set

The following variables are excluded from model training:

- Underwriting variables  
Fields such as `verification_status` and `is_inc_v` reflect actions taken after application submission and are therefore outside the defined prediction boundary.

- Platform pricing and risk assignment variables  
Variables such as `int_rate`, `grade`, `sub_grade`, and `installment` are determined during underwriting and pricing. These reflect LendingClub’s internal risk assessment and are excluded from training to avoid circularity.

- Platform workflow variables  
Internal listing or review fields occur after submission and do not represent borrower risk characteristics.

- Post-origination servicing variables  
Payment history, recoveries, outstanding balances, and related servicing metrics are only available after loan issuance and would introduce data leakage if included.

- External risk estimates  
`expDefaultRate` represents LendingClub’s expected default estimate and is excluded to preserve model independence.

- Structural and non-informative variables  
Identifiers, URLs, constant columns, and fully null columns are removed during ETL.

---

### Benchmark Variables Retained for Validation

- A subset of platform-assigned risk signals (`grade`, `sub_grade`, `int_rate`, and `installment`) will be retained in the cleaned dataset for benchmarking purposes in the validation notebook.

- These variables will not be used for training but will serve as reference points when evaluating the performance of the submission-time model against LendingClub’s origination-time risk signals.

---

### Target Variable

`loan_status` is retained exclusively for outcome definition and will not be used as a predictor.

Excluded variables will be removed from the training feature set before the EDA phase, while benchmark variables will remain available for validation analysis.


## Feature Classification Overview – Application Submission Prediction Point

The table below summarizes the treatment of variables based on the defined prediction point at loan application submission and the observed structure of the 2007–2014 training dataset.

| Column | Category | Action | Rationale |
|--------|----------|--------|-----------|
| `annual_inc`, `dti` | Application input | Keep | Provided at submission |
| `loan_amnt`, `term`, `purpose` | Application input | Keep | Provided at submission |
| `home_ownership`, `emp_length` | Application input | Keep | Provided at submission |
| `fico_range_low`, `fico_range_high` | Credit snapshot | Keep | Credit bureau state at submission |
| `earliest_cr_line` | Credit snapshot | Keep | Historical credit information |
| `open_acc`, `total_acc` | Credit snapshot | Keep | Credit file state |
| `inq_last_6mths` | Credit snapshot | Keep | Recent inquiry count |
| `delinq_2yrs` | Credit snapshot | Keep | Delinquency history |
| `pub_rec` | Credit snapshot | Keep | Public record count |
| `collections_12_mths_ex_med` | Credit snapshot | Keep | Recent collections |
| `revol_bal`, `revol_util` | Credit snapshot | Keep | Revolving credit usage |
| `acc_now_delinq` | Credit snapshot | Keep | Current delinquency count |
| `tot_cur_bal`, `tot_coll_amt` | Credit snapshot | Keep | Credit balance metrics |
| `mths_since_last_delinq` | Credit timing | Keep (with transformation) | Null indicates no prior event; encoded with indicator and sentinel value |
| `mths_since_last_major_derog` | Credit timing | Keep (with transformation) | Null indicates no prior event; encoded with indicator and sentinel value |
| `mths_since_last_record` | Credit timing | Keep (with transformation) | Null indicates no prior event; encoded with indicator and sentinel value |
| `loan_status` | Target | Keep (Target Only) | Defines outcome; not used as predictor |
| `grade`, `sub_grade` | Platform risk signal | Retain (Benchmark Only) | Assigned at origination; excluded from training; used for validation comparison |
| `int_rate` | Pricing signal | Retain (Benchmark Only) | Reflects underwriting assessment; excluded from training; used for benchmarking |
| `installment` | Pricing-derived | Retain (Benchmark Only) | Derived from interest rate and principal; excluded from training; used for benchmarking |
| `verification_status`, `is_inc_v` | Underwriting variable | Exclude | Determined during underwriting; outside defined submission boundary |
| `funded_amnt`, `funded_amnt_inv` | Platform decision | Exclude | Determined during funding decision |
| `issue_d` | Platform decision | Exclude | Funding date; post submission |
| `initial_list_status` | Platform decision | Exclude | Listing outcome |
| `last_credit_pull_d` | Post-submission update | Exclude | Updated after submission |
| `last_fico_range_high`, `last_fico_range_low` | Post-submission update | Exclude | Subsequent credit updates |
| `total_pymnt`, `total_pymnt_inv` | Post-origination | Exclude | Payment history; not available at submission |
| `total_rec_prncp`, `total_rec_int`, `total_rec_late_fee` | Post-origination | Exclude | Cashflows after issuance |
| `recoveries`, `collection_recovery_fee` | Post-origination | Exclude | Post charge-off recovery information |
| `out_prncp`, `out_prncp_inv` | Post-origination | Exclude | Outstanding principal after issuance |
| `last_pymnt_d`, `next_pymnt_d` | Post-origination | Exclude | Servicing timeline variables |
| `last_pymnt_amnt` | Post-origination | Exclude | Payment received after issuance |
| `pymnt_plan` | Post-origination | Exclude | Servicing decision variable |
| `expDefaultRate` | External risk estimate | Exclude | Platform expected default estimate |
| `annual_inc_joint`, `dti_joint`, `verification_status_joint` | Fully null (2007–2014) | Drop | No observations in training data |
| `open_acc_6m`, `open_il_6m`, `open_il_12m`, `open_il_24m` | Fully null (2007–2014) | Drop | No observations in training data |
| `mths_since_rcnt_il`, `total_bal_il`, `il_util` | Fully null (2007–2014) | Drop | No observations in training data |
| `open_rv_12m`, `open_rv_24m`, `all_util` | Fully null (2007–2014) | Drop | No observations in training data |
| `inq_fi`, `total_cu_tl`, `inq_last_12m`, `max_bal_bc` | Fully null (2007–2014) | Drop | No observations in training data |
| `id`, `member_id` | Identifier | Drop | Unique identifiers; no predictive value |
| `url` | Identifier | Drop | Platform listing URL; no predictive value |
| `Unnamed: 0` | Structural | Drop | Index artifact |
| `policy_code`, `application_type` | Constant | Drop | No variance in training data |
| `desc`, `emp_title`, `title` | Free-text field | Drop |  High cardinality; Unstructured text; excluded from structured modeling scope |
| `addr_state`, `zip_code` | Application input | Drop | Removed to reduce proxy discrimination risk and improve portability across regulatory environments |


## Initial Data Inspection – Test Set (2015)

I apply the same inspection procedure to the 2015 test dataset to confirm that its structure is consistent with the training data and that the ETL rules can be applied without introducing train–test mismatches.

In [10]:
# Loading the csv file into pandas dataframes
df_raw_test_data = pd.read_csv(raw_test_data_file)
df_raw_test_data.head(5)

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_30888\3316187366.py:2: DtypeWarning: Columns (0: desc, 1: verification_status_joint) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw_test_data = pd.read_csv(raw_test_data_file)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,68444620,73334399,35000,35000,35000.0,60 months,11.99,778.38,C,C1,...,35367.0,49.3,0.0,1.0,5020.0,40.1,52200.0,1.0,4.0,0.0
1,68547583,73437441,8650,8650,8650.0,36 months,5.32,260.50,A,A1,...,24041.0,88.8,0.0,3.0,3081.0,57.9,26800.0,1.0,0.0,5.0
2,67849662,72708407,4225,4225,4225.0,36 months,14.85,146.16,C,C5,...,3830.0,21.9,0.0,0.0,367.0,22.4,4300.0,0.0,0.0,0.0
3,68506885,73396712,10000,10000,10000.0,60 months,11.99,222.40,C,C1,...,35354.0,75.5,1.0,1.0,3118.0,67.4,14200.0,1.0,1.0,1.0
4,68341763,72928789,20000,20000,20000.0,60 months,10.78,432.66,B,B4,...,10827.0,72.8,0.0,2.0,2081.0,64.7,14000.0,2.0,5.0,1.0


In [11]:
# Initial inspection of the raw test data
inspection_summary_test = initial_inspection(df_raw_test_data, project_log_file)

In [12]:
# Show dataset structure and memory usage of the raw test data
rows, cols = inspection_summary_test["shape"]
memory_mb = inspection_summary_test["memory_usage_mb"]

pd.DataFrame({
    "Rows": [rows],
    "Columns": [cols],
    "Memory Usage (MB)": [round(memory_mb, 2)]
})


,Rows,Columns,Memory Usage (MB)
0,421094,74,322.85


In [13]:
# Show feature-level summary of the raw training data, sorted by percentage of missing values
with pd.option_context("display.max_rows", None,
                       "display.max_columns", None):
    
    display(
        inspection_summary_test["feature_summary"]
        .sort_values("null_percent", ascending=False)
    )

,dtype,n_unique,null_percent,is_numeric,is_object,is_mixed_type,is_numeric_object,is_fully_null,is_constant
desc,str,35,99.99,False,True,False,False,False,False
annual_inc_joint,float64,309,99.88,True,False,False,False,False,False
dti_joint,float64,450,99.88,True,False,False,False,False,False
verification_status_joint,str,4,99.88,False,True,False,False,False,False
il_util,float64,1273,95.58,True,False,False,False,False,False
mths_since_rcnt_il,float64,202,95.06,True,False,False,False,False,False
open_rv_24m,float64,29,94.92,True,False,False,False,False,False
open_acc_6m,float64,14,94.92,True,False,False,False,False,False
all_util,float64,1129,94.92,True,False,False,False,False,False
inq_last_12m,float64,30,94.92,True,False,False,False,False,False


In [14]:
# Access the feature summary from the inspection
feature_summary_test_df = inspection_summary_test["feature_summary"]

# Build a list of structural issues
structural_issues_records = []

for column_name in feature_summary_test_df.index:
    if feature_summary_test_df.loc[column_name, "is_fully_null"]:
        structural_issues_records.append({"Column": column_name, "Issue": "100% Null"})
    elif feature_summary_test_df.loc[column_name, "null_percent"] > 50:
        structural_issues_records.append({"Column": column_name, "Issue": "High Missing (>50%)"})
    elif feature_summary_test_df.loc[column_name, "is_constant"]:
        structural_issues_records.append({"Column": column_name, "Issue": "Constant"})
    elif feature_summary_test_df.loc[column_name, "is_mixed_type"]:
        structural_issues_records.append({"Column": column_name, "Issue": "Mixed Type"})
    elif feature_summary_test_df.loc[column_name, "is_numeric_object"]:
        structural_issues_records.append({"Column": column_name, "Issue": "Numeric Stored as Object"})

# Convert to DataFrame
structural_issues_test_df = pd.DataFrame(structural_issues_records)

# Define severity order
severity_order = [
    "100% Null",
    "High Missing (>50%)",
    "Constant",
    "Mixed Type",
    "Numeric Stored as Object"
]

# Sort by severity and then by column name
structural_issues_test_df["Issue"] = pd.Categorical(
    structural_issues_test_df["Issue"],
    categories=severity_order,
    ordered=True
)
structural_issues_test_df = structural_issues_test_df.sort_values(
    ["Issue", "Column"]
).reset_index(drop=True)

# Display the sorted structural issues
display(structural_issues_test_df)


,Column,Issue
0,all_util,High Missing (>50%)
1,annual_inc_joint,High Missing (>50%)
2,desc,High Missing (>50%)
3,dti_joint,High Missing (>50%)
4,il_util,High Missing (>50%)
5,inq_fi,High Missing (>50%)
6,inq_last_12m,High Missing (>50%)
7,max_bal_bc,High Missing (>50%)
8,mths_since_last_major_derog,High Missing (>50%)
9,mths_since_last_record,High Missing (>50%)


In [15]:
# In the initial inspection, we found that the training data has 1 additional column compared to the test data:
set(df_raw_training_data.columns) - set(df_raw_test_data.columns)


{'Unnamed: 0'}

## Test Set Inspection – Structural Consistency Check

The 2015 test dataset contains 74 columns compared to 75 in the training data. The only structural difference is the absence of the `Unnamed: 0` column in the test set. This column is a CSV export artifact and is removed during ETL in any case, so it does not affect downstream processing.

A more substantive difference is observed in several credit bureau variables. In the 2007–2014 training data, 17 columns are fully null, indicating that these features were not populated during that period. In the 2015 dataset, some of these same columns contain values but exhibit very high missing rates.

Because the model is trained exclusively on 2007–2014 data, features that are entirely absent in the training period cannot be meaningfully incorporated at prediction time. To maintain temporal consistency and prevent unseen-feature bias, these columns are removed from both the training and test datasets.

Aside from these expected temporal differences in feature availability, the schema, data types, and structural characteristics of the test set align with the training data. The same transformation rules and feature eligibility decisions will therefore be applied consistently across both datasets.


## Data Transformation and Cleaning

Both the training (2007–2014) and testing (2015) datasets are transformed using the feature eligibility rules defined above. The same transformation logic is applied to both datasets to ensure identical schemas prior to modeling.

The cleaning phase performs the following steps:

- Removal of structural artifacts, constant columns, and features excluded from the submission-time feature set.  
- Alignment of the test dataset to the training feature space, including removal of columns that are absent in the training period.  
- Creation of a stable internal `row_id` to preserve record-level traceability across modeling and validation outputs.  
- Encoding of credit timing variables (`mths_since_last_delinq`, `mths_since_last_major_derog`, `mths_since_last_record`) using a binary event indicator and a sentinel value (9999) to represent structural absence.  
- Inspection and normalization of remaining string-based features after structural column removal. Where necessary, categorical formatting is standardized and numeric or date values stored as strings are converted to appropriate data types to ensure consistency prior to exploratory analysis.

Two dataset variants are produced:

1. A modeling dataset containing only submission-time features and the target variable, used for EDA and model training.  
2. An extended dataset retaining benchmark variables for validation and comparative analysis.

This ensures that the modeling pipeline operates on a consistent, temporally aligned feature space while preserving selected platform signals for evaluation purposes.


In [16]:
# Create row identifier for training data
df_clean_training_data = create_row_identifier(
    df_raw_training_data, 
    id_column_name="row_id", 
    log_file=project_log_file)

In [17]:
# Create row identifier for test data
df_clean_test_data = create_row_identifier(
    df_raw_test_data, 
    id_column_name="row_id", 
    log_file=project_log_file)

In [18]:
# Parameters for sentinel encoding of credit history features.
credit_recency_columns = [
    "mths_since_last_delinq",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

SENTINEL_VALUE = 9999

# Columns to drop for clean_datasets
structurally_drop_columns = [
    "annual_inc_joint", "dti_joint", "verification_status_joint", "open_acc_6m", "open_il_6m", "open_il_12m", 
    "open_il_24m","mths_since_rcnt_il", "total_bal_il", "il_util","open_rv_12m", "open_rv_24m", "all_util",
    "inq_fi", "total_cu_tl", "inq_last_12m", "max_bal_bc","id", "member_id","url", "Unnamed: 0", "policy_code", 
    "application_type", "desc", "emp_title", "title", "addr_state", "zip_code"
]

#columns to drop for modeling datasets (after cleaning)
excluded_columns = [
    "verification_status", "is_inc_v", "funded_amnt", "funded_amnt_inv", "issue_d", "initial_list_status",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "out_prncp", "out_prncp_inv", "last_pymnt_d", "next_pymnt_d", "last_pymnt_amnt", "pymnt_plan", "expDefaultRate"
]

benchmark_columns = ["grade", "sub_grade", "int_rate", "installment"]



In [19]:
# Apply sentinel encoding to the credit recency columns in the training data
df_clean_training_data = apply_credit_sentinel_encoding(
    df_clean_training_data,
    credit_recency_columns,
    sentinel_value=SENTINEL_VALUE,
    log_file=project_log_file)

In [20]:
# Apply sentinel encoding to the credit recency columns in the test data
df_clean_test_data = apply_credit_sentinel_encoding(
    df_clean_test_data,
    credit_recency_columns,
    sentinel_value=SENTINEL_VALUE,
    log_file=project_log_file)

In [21]:
# Drop columns for cleaning training dataset
df_clean_training_data = drop_columns_with_logging(
    dataframe=df_clean_training_data,
    columns_to_drop=structurally_drop_columns,
    dataset_name="train_clean",
    log_file=project_log_file,
    errors='ignore'
    )

df_clean_training_data.head(5)

,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,B,B2,10+ years,...,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,C,C4,< 1 year,...,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,C,C5,10+ years,...,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,C,C1,10+ years,...,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,B,B5,1 year,...,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


In [22]:
# Drop columns for cleaning test dataset
df_clean_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=structurally_drop_columns,
    dataset_name="test_clean",
    log_file=project_log_file,
    errors='ignore'
    )

df_clean_test_data.head(5)

,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,...,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,C,C1,10+ years,...,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,A,A1,< 1 year,...,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,C,C5,5 years,...,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,C,C1,10+ years,...,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,B,B4,10+ years,...,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


In [23]:
# Inspecting string columns in the clean datasets
training_string_audit = audit_string_columns(
    df_clean_training_data,
    sample_size=15,
    log_file=project_log_file
)

test_string_audit = audit_string_columns(
    df_clean_test_data,
    sample_size=15,
    log_file=project_log_file
)

# Display training string audit
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    
    display(
        training_string_audit.sort_values("unique_count_non_null", ascending=False)
    )

    
# Display test string audit
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    
    display(
        test_string_audit.sort_values("unique_count_non_null", ascending=False)
    )


,column_name,dtype,unique_count_including_null,unique_count_non_null,null_percent,sample_values
0,earliest_cr_line,str,665,664,0.01,"[Jan-85, Apr-99, Nov-01, Feb-96, Jan-96, Nov-0..."
1,last_credit_pull_d,str,104,103,0.01,"[Jan-16, Sep-13, Jan-15, Sep-15, Dec-14, Aug-1..."
2,next_pymnt_d,str,101,100,48.73,"[Feb-16, Jan-16, Sep-13, Feb-14, May-14, Jun-1..."
3,last_pymnt_d,str,99,98,0.08,"[Jan-15, Apr-13, Jun-14, Jan-16, Apr-12, Nov-1..."
4,issue_d,str,91,91,0.00,"[Dec-11, Nov-11, Oct-11, Sep-11, Aug-11, Jul-1..."
5,sub_grade,str,35,35,0.00,"[B2, C4, C5, C1, B5, A4, E1, F2, C3, B1, D1, A..."
6,purpose,str,14,14,0.00,"[credit_card, car, small_business, other, wedd..."
7,emp_length,str,12,11,4.51,"[10+ years, < 1 year, 1 year, 3 years, 8 years..."
8,loan_status,str,9,9,0.00,"[Fully Paid, Charged Off, Current, Default, La..."
9,grade,str,7,7,0.00,"[B, C, A, E, F, D, G]"


,column_name,dtype,unique_count_including_null,unique_count_non_null,null_percent,sample_values
0,earliest_cr_line,str,668,668,0.00,"[Feb-90, Jul-01, Jul-11, Dec-98, Aug-00, Nov-0..."
1,sub_grade,str,35,35,0.00,"[C1, A1, C5, B4, B3, A4, C4, C3, F1, D3, B1, E..."
2,purpose,str,14,14,0.00,"[home_improvement, credit_card, debt_consolida..."
3,last_credit_pull_d,str,15,14,0.00,"[Jan-16, Dec-15, Nov-15, Sep-15, Oct-15, Aug-1..."
4,last_pymnt_d,str,14,13,4.10,"[Jan-16, Dec-15, Nov-15, Oct-15, Sep-15, Aug-1..."
5,issue_d,str,12,12,0.00,"[Dec-15, Nov-15, Oct-15, Sep-15, Aug-15, Jul-1..."
6,emp_length,str,12,11,5.66,"[10+ years, < 1 year, 5 years, 3 years, 4 year..."
7,loan_status,str,8,8,0.00,"[Issued, Current, Fully Paid, In Grace Period,..."
8,grade,str,7,7,0.00,"[C, A, B, F, D, E, G]"
9,home_ownership,str,4,4,0.00,"[MORTGAGE, RENT, OWN, ANY]"


In [24]:
# Compare the string columns between training and test datasets
audit_columns_to_compare = [
    "column_name",
    "dtype",
    "unique_count_non_null",
    "unique_count_including_null",
    "null_percent",
]

training_audit_subset = training_string_audit[audit_columns_to_compare].copy()
test_audit_subset = test_string_audit[audit_columns_to_compare].copy()

training_audit_subset = training_audit_subset.rename(columns={
    "dtype": "dtype_train",
    "unique_count_non_null": "unique_non_null_train",
    "unique_count_including_null": "unique_including_null_train",
    "null_percent": "null_percent_train",
})

test_audit_subset = test_audit_subset.rename(columns={
    "dtype": "dtype_test",
    "unique_count_non_null": "unique_non_null_test",
    "unique_count_including_null": "unique_including_null_test",
    "null_percent": "null_percent_test",
})

string_audit_comparison = (
    training_audit_subset
    .merge(test_audit_subset, on="column_name", how="outer")
    .sort_values(by="column_name")
    .reset_index(drop=True)
)

string_audit_comparison["unique_non_null_diff"] = (
    string_audit_comparison["unique_non_null_train"].fillna(0).astype(int)
    - string_audit_comparison["unique_non_null_test"].fillna(0).astype(int)
)

string_audit_comparison


,column_name,dtype_train,unique_non_null_train,unique_including_null_train,null_percent_train,dtype_test,unique_non_null_test,unique_including_null_test,null_percent_test,unique_non_null_diff
0,earliest_cr_line,str,664,665,0.01,str,668,668,0.00,-4
1,emp_length,str,11,12,4.51,str,11,12,5.66,0
2,grade,str,7,7,0.00,str,7,7,0.00,0
3,home_ownership,str,6,6,0.00,str,4,4,0.00,2
4,initial_list_status,str,2,2,0.00,str,2,2,0.00,0
5,issue_d,str,91,91,0.00,str,12,12,0.00,79
6,last_credit_pull_d,str,103,104,0.01,str,14,15,0.00,89
7,last_pymnt_d,str,98,99,0.08,str,13,14,4.10,85
8,loan_status,str,9,9,0.00,str,8,8,0.00,1
9,next_pymnt_d,str,100,101,48.73,str,3,4,6.12,97


In [25]:
string_columns_with_differences = string_audit_comparison[
    (string_audit_comparison["dtype_train"] != string_audit_comparison["dtype_test"])
    | (string_audit_comparison["unique_non_null_train"] != string_audit_comparison["unique_non_null_test"])
    | (string_audit_comparison["null_percent_train"].round(2) != string_audit_comparison["null_percent_test"].round(2))
].copy()

string_columns_with_differences


,column_name,dtype_train,unique_non_null_train,unique_including_null_train,null_percent_train,dtype_test,unique_non_null_test,unique_including_null_test,null_percent_test,unique_non_null_diff
0,earliest_cr_line,str,664,665,0.01,str,668,668,0.00,-4
1,emp_length,str,11,12,4.51,str,11,12,5.66,0
3,home_ownership,str,6,6,0.00,str,4,4,0.00,2
5,issue_d,str,91,91,0.00,str,12,12,0.00,79
6,last_credit_pull_d,str,103,104,0.01,str,14,15,0.00,89
7,last_pymnt_d,str,98,99,0.08,str,13,14,4.10,85
8,loan_status,str,9,9,0.00,str,8,8,0.00,1
9,next_pymnt_d,str,100,101,48.73,str,3,4,6.12,97


In [26]:
columns_to_drill_down = (
    string_columns_with_differences["column_name"]
    .dropna()
    .astype(str)
    .tolist()
)

categorical_value_differences = {}

for column_name in columns_to_drill_down:
    categorical_value_differences[column_name] = compare_categorical_column_values(
        training_dataframe=df_clean_training_data,
        test_dataframe=df_clean_test_data,
        column_name=column_name,
        log_file=project_log_file,
    )


In [27]:
summary_records = []

for column_name in columns_to_drill_down:
    comparison_dataframe = compare_categorical_column_values(
        training_dataframe=df_clean_training_data,
        test_dataframe=df_clean_test_data,
        column_name=column_name,
        log_file=project_log_file,
    )

    categorical_value_differences[column_name] = comparison_dataframe

    missing_in_training = (~comparison_dataframe["present_in_training"]).sum()
    missing_in_test = (~comparison_dataframe["present_in_test"]).sum()

    summary_records.append({
        "column_name": column_name,
        "categories_total": len(comparison_dataframe),
        "only_in_test": int(missing_in_training),
        "only_in_training": int(missing_in_test),
    })

df_category_diff_summary = (
    pd.DataFrame(summary_records)
    .sort_values(["only_in_test", "only_in_training"], ascending=False)
    .reset_index(drop=True)
)

df_category_diff_summary


,column_name,categories_total,only_in_test,only_in_training
0,earliest_cr_line,697,33,29
1,issue_d,103,12,91
2,loan_status,10,1,2
3,next_pymnt_d,100,0,97
4,last_credit_pull_d,103,0,89
5,last_pymnt_d,98,0,85
6,home_ownership,6,0,2
7,emp_length,11,0,0


In [28]:
# Log categorical differences across training and test datasets
categorical_value_differences = {}

for column_name in columns_to_drill_down:
    categorical_value_differences[column_name] = compare_categorical_column_values(
        training_dataframe=df_clean_training_data,
        test_dataframe=df_clean_test_data,
        column_name=column_name,
        log_file=project_log_file,
    )

log_category_differences(
    categorical_value_differences=categorical_value_differences,
    log_file=project_log_file,
)


## String Column Treatment Overview

The table below summarizes how remaining string-based columns are handled during transformation.  
Each column is classified by:

- **Transformation Action** → how it is technically processed.
- **Training Eligibility** → whether it is used for modeling, excluded, or dropped.

Category differences between the 2007–2014 training data and the 2015 test data were reviewed and found to reflect time-window effects rather than structural inconsistencies.

| Column | Category | Transformation Action | Training Eligibility | Rationale |
|---------|------------|------------------------|----------------------|------------|
| `term` | Structured categorical | Strip whitespace → extract numeric term (36 / 60) → rename to `term_months` → convert to int | Keep | Submission-time contractual input |
| `emp_length` | Ordinal categorical | Map to integer 0–10 (`<1` → 0, `10+` → 10); retain NaN structure | Keep | Missing values represent unknown employment tenure |
| `home_ownership` | Categorical | Standardize casing; map `NONE` → `OTHER`; cast to category | Keep | Submission-time borrower attribute |
| `purpose` | Categorical | Standardize casing; cast to category | Keep | Submission-time borrower input |
| `grade` | Platform risk signal | Standardize casing; cast to category | Exclude (Benchmark Only) | Assigned during underwriting |
| `sub_grade` | Platform risk signal | Standardize casing; cast to category | Exclude (Benchmark Only) | Granular underwriting signal |
| `verification_status` | Underwriting variable | Standardize casing; cast to category | Exclude | Determined after submission |
| `initial_list_status` | Platform workflow | Standardize casing; cast to category | Exclude | Post-submission listing outcome |
| `pymnt_plan` | Servicing variable | Map `n` → 0, `y` → 1 | Exclude | Post-origination servicing indicator |
| `issue_d` | Post-submission date | Convert from month-year string to datetime | Exclude | Outside submission boundary |
| `last_credit_pull_d` | Post-submission date | Convert from month-year string to datetime | Exclude | Subsequent bureau update |
| `last_pymnt_d` | Post-origination date | Convert from month-year string to datetime | Exclude | Servicing information |
| `next_pymnt_d` | Post-origination date | Convert from month-year string to datetime | Exclude | Servicing information |
| `loan_status` | Target | No transformation at this stage | Keep (Target Only) | Outcome variable; modeling cohort defined later |

### Structural Principles

All observed train/test category differences are attributable to temporal coverage rather than schema drift.  
The resulting feature space aligns strictly with the submission-time prediction boundary.


In [29]:
# -----------------------------------
# Columns grouped by transformation type
# -----------------------------------

# 1. Text normalization (strip + lowercase + cast to category)
# Used to standardize categorical values and ensure train/test alignment.
categorical_normalization_columns = [
    "home_ownership",
    "purpose",
    "grade",
    "sub_grade",
    "verification_status",
    "initial_list_status",
    "loan_status",
]

# 2. Deterministic category remapping
# Used to consolidate semantically equivalent labels.
special_categorical_mappings = {
    "home_ownership": {
        "none": "other",
        "any": "other",
    }
}

# 3. Ordinal encoding
# Used for ordered categories with intrinsic rank.
ordinal_encoding_columns = {
    "emp_length": {
        "< 1 year": 0,
        "1 year": 1,
        "2 years": 2,
        "3 years": 3,
        "4 years": 4,
        "5 years": 5,
        "6 years": 6,
        "7 years": 7,
        "8 years": 8,
        "9 years": 9,
        "10+ years": 10
    }
}

# 4. Structured extraction
# Convert contract term to numeric month representation.
term_column = "term"
term_new_name = "term_months"

# 5. Binary encoding
# Convert binary categorical indicators to numeric form.
binary_encoding_columns = {
    "pymnt_plan": {
        "n": 0,
        "y": 1
    }
}

# 6. Month-Year datetime conversion
# Convert month-year strings to datetime format.
month_year_datetime_columns = [
    "issue_d",
    "last_credit_pull_d",
    "last_pymnt_d",
    "next_pymnt_d",
    "earliest_cr_line",
]

# 7. Explicit mapping for listing type
# Replace abbreviated codes with semantic labels.
initial_list_status_mapping = {
    "w": "whole",
    "f": "fractional"
}


In [30]:
# 1. Normalize text categories (casing/whitespace)
df_clean_training_data = normalize_text_columns(
    dataframe=df_clean_training_data,
    columns_to_normalize=categorical_normalization_columns,
    log_file=project_log_file,
)

df_clean_test_data = normalize_text_columns(
    dataframe=df_clean_test_data,
    columns_to_normalize=categorical_normalization_columns,
    log_file=project_log_file,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,n,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,f,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,n,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,n,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,f,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,n,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,f,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,n,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,f,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,n,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,w,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,n,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,w,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,n,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,w,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,n,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,w,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,n,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,w,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                         str
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                         str
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [31]:
# 2. Home ownership alignment
df_clean_training_data = normalize_home_ownership(df_clean_training_data, log_file=project_log_file)
df_clean_test_data = normalize_home_ownership(df_clean_test_data, log_file=project_log_file)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)
Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,n,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,f,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,n,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,n,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,f,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,n,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,f,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,n,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,f,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,n,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,w,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,n,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,w,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,n,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,w,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,n,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,w,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,n,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,w,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [32]:
# 3. initial_list_status mapping (w/f -> words)
df_clean_training_data = apply_categorical_mapping(
    dataframe=df_clean_training_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log_file=project_log_file,
    allow_unmapped_values=False,
)

df_clean_test_data = apply_categorical_mapping(
    dataframe=df_clean_test_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log_file=project_log_file,
    allow_unmapped_values=False,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,n,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,n,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,n,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,n,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,n,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,n,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,n,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,n,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,n,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,n,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                             str
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [33]:
# 4. pymnt_plan binary
df_clean_training_data = apply_categorical_mapping(
    dataframe=df_clean_training_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log_file=project_log_file,
    allow_unmapped_values=False,
)

df_clean_test_data = apply_categorical_mapping(
    dataframe=df_clean_test_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log_file=project_log_file,
    allow_unmapped_values=False,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,0,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,0,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,0,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,0,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,0,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,0,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,0,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,0,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,0,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,0,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [34]:
# 5. Term parsing + rename
# (Implement as a dedicated function if you want logging; otherwise do it inline)
df_clean_training_data[term_new_name] = (
    df_clean_training_data[term_column].astype("string").str.strip().str.extract(r"(\d+)")[0].astype("Int16")
)
df_clean_test_data[term_new_name] = (
    df_clean_test_data[term_column].astype("string").str.strip().str.extract(r"(\d+)")[0].astype("Int16")
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 52) | Test shape: (421094, 52)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,0,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,0,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,0,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,0,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,0,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,0,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,0,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,0,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,0,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,0,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [35]:
# 6. emp_length ordinal
df_clean_training_data = transform_emp_length(df_clean_training_data, log_file=project_log_file)
df_clean_test_data = transform_emp_length(df_clean_test_data, log_file=project_log_file)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 53) | Test shape: (421094, 53)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,Dec-11,fully_paid,0,credit_card,27.65,0.0,Jan-85,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,Dec-11,charged_off,0,car,1.00,0.0,Apr-99,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,Dec-11,fully_paid,0,small_business,8.72,0.0,Nov-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,Dec-11,fully_paid,0,other,20.00,0.0,Feb-96,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,Dec-11,current,0,other,17.94,0.0,Jan-96,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,Dec-15,issued,0,home_improvement,6.46,0.0,Feb-90,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,Dec-15,issued,0,credit_card,7.28,0.0,Jul-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,Dec-15,issued,0,debt_consolidation,15.22,2.0,Jul-11,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,Dec-15,issued,0,debt_consolidation,31.04,0.0,Dec-98,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Jan-16,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,Dec-15,issued,0,home_improvement,10.78,0.0,Aug-00,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,Jan-16,Dec-15,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

row_id                               int64
loan_amnt                            int64
funded_amnt                          int64
funded_amnt_inv                    float64
term                                   str
int_rate                           float64
installment                        float64
grade                                  str
sub_grade                              str
emp_length                             str
home_ownership                      string
annual_inc                         float64
verification_status                    str
issue_d                                str
loan_status                            str
pymnt_plan                           int64
purpose                                str
dti                                float64
delinq_2yrs                        float64
earliest_cr_line                       str
inq_last_6mths                     float64
mths_since_last_delinq             float64
mths_since_last_record             float64
open_acc   

In [36]:
# 7. Month-year datetime conversions (including excluded/benchmark dates, per your choice)
df_clean_training_data = convert_month_year_columns_to_datetime(
    dataframe=df_clean_training_data,
    column_names=month_year_datetime_columns,
    log_file=project_log_file,
)
df_clean_test_data = convert_month_year_columns_to_datetime(
    dataframe=df_clean_test_data,
    column_names=month_year_datetime_columns,
    log_file=project_log_file,
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 53) | Test shape: (421094, 53)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,36 months,10.65,162.87,b,b2,10+ years,rent,24000.0,verified,2011-12-01,fully_paid,0,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,60 months,15.27,59.83,c,c4,< 1 year,rent,30000.0,source_verified,2011-12-01,charged_off,0,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,36 months,15.96,84.33,c,c5,10+ years,rent,12252.0,not_verified,2011-12-01,fully_paid,0,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,36 months,13.49,339.31,c,c1,10+ years,rent,49200.0,source_verified,2011-12-01,fully_paid,0,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,60 months,12.69,67.79,b,b5,1 year,rent,80000.0,source_verified,2011-12-01,current,0,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,60 months,11.99,778.38,c,c1,10+ years,mortgage,128000.0,source_verified,2015-12-01,issued,0,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,36 months,5.32,260.50,a,a1,< 1 year,mortgage,100000.0,not_verified,2015-12-01,issued,0,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,36 months,14.85,146.16,c,c5,5 years,rent,35000.0,source_verified,2015-12-01,issued,0,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,60 months,11.99,222.40,c,c1,10+ years,rent,42500.0,not_verified,2015-12-01,issued,0,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,60 months,10.78,432.66,b,b4,10+ years,mortgage,63000.0,not_verified,2015-12-01,issued,0,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2015-12-01,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
term                                          str
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
emp_length                                    str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
term                                          str
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
emp_length                                    str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]


In [37]:
# 8. We drop old columns term and emp_legth
df_clean_training_data = drop_columns_with_logging(
    dataframe=df_clean_training_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="train_clean",
    log_file=project_log_file,
    errors='ignore'
)

df_clean_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="test_clean",
    log_file=project_log_file,
    errors='ignore'
)

print(
    f"Train shape: {df_clean_training_data.shape} | "
    f"Test shape: {df_clean_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)

Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,10.65,162.87,b,b2,rent,24000.0,verified,2011-12-01,fully_paid,0,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,15.27,59.83,c,c4,rent,30000.0,source_verified,2011-12-01,charged_off,0,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,15.96,84.33,c,c5,rent,12252.0,not_verified,2011-12-01,fully_paid,0,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,13.49,339.31,c,c1,rent,49200.0,source_verified,2011-12-01,fully_paid,0,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,12.69,67.79,b,b5,rent,80000.0,source_verified,2011-12-01,current,0,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,11.99,778.38,c,c1,mortgage,128000.0,source_verified,2015-12-01,issued,0,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,5.32,260.50,a,a1,mortgage,100000.0,not_verified,2015-12-01,issued,0,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,14.85,146.16,c,c5,rent,35000.0,source_verified,2015-12-01,issued,0,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,11.99,222.40,c,c1,rent,42500.0,not_verified,2015-12-01,issued,0,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,10.78,432.66,b,b4,mortgage,63000.0,not_verified,2015-12-01,issued,0,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2015-12-01,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                         str
sub_grade                                     str
home_ownership                             string
annual_inc                                float64
verification_status                           str
issue_d                            datetime64[ns]
loan_status                                   str
pymnt_plan                                  int64
purpose                                       str
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


In [38]:
# 9. Standardize dtypes for categorical columns
log = get_logger(project_log_file)

categorical_columns_to_cast = [
    "purpose",
    "home_ownership",
    "verification_status",
    "initial_list_status",
    "grade",
    "sub_grade",
    "loan_status",
]

try:
    log("Casting categorical columns: string -> category (train/test)")

    for column_name in categorical_columns_to_cast:
        if column_name not in df_clean_training_data.columns:
            log(f"Train missing categorical column (skipped): {column_name}")
            continue
        if column_name not in df_clean_test_data.columns:
            log(f"Test missing categorical column (skipped): {column_name}")
            continue

        # Convert both to pandas string first (keeps <NA> as missing)
        df_clean_training_data[column_name] = df_clean_training_data[column_name].astype("string")
        df_clean_test_data[column_name] = df_clean_test_data[column_name].astype("string")

        # Build a shared category space across train/test (prevents drift)
        train_values = df_clean_training_data[column_name].dropna().unique().tolist()
        test_values = df_clean_test_data[column_name].dropna().unique().tolist()
        combined_categories = sorted(set(train_values) | set(test_values))

        df_clean_training_data[column_name] = pd.Categorical(
            df_clean_training_data[column_name],
            categories=combined_categories
        )
        df_clean_test_data[column_name] = pd.Categorical(
            df_clean_test_data[column_name],
            categories=combined_categories
        )

        log(
            f"Casted '{column_name}' to category | "
            f"train_unique={len(train_values)} | test_unique={len(test_values)} | combined={len(combined_categories)}"
        )

except Exception as error:
    log(f"Error during categorical dtype casting: {error}")
    raise

print(f"Train shape: {df_clean_training_data.shape} | Test shape: {df_clean_test_data.shape}")

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_clean_training_data.head(5))
    display(df_clean_test_data.head(5))

display(df_clean_training_data.dtypes)
display(df_clean_test_data.dtypes)


Train shape: (466285, 51) | Test shape: (421094, 51)


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,5000,4975.0,10.65,162.87,b,b2,rent,24000.0,verified,2011-12-01,fully_paid,0,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,fractional,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,2500,2500.0,15.27,59.83,c,c4,rent,30000.0,source_verified,2011-12-01,charged_off,0,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,fractional,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,2400,2400.0,15.96,84.33,c,c5,rent,12252.0,not_verified,2011-12-01,fully_paid,0,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,fractional,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,10000,10000.0,13.49,339.31,c,c1,rent,49200.0,source_verified,2011-12-01,fully_paid,0,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,fractional,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,3000,3000.0,12.69,67.79,b,b5,rent,80000.0,source_verified,2011-12-01,current,0,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,fractional,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,grade,sub_grade,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,35000,35000.0,11.99,778.38,c,c1,mortgage,128000.0,source_verified,2015-12-01,issued,0,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,whole,35000.0,35000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,8650,8650.0,5.32,260.50,a,a1,mortgage,100000.0,not_verified,2015-12-01,issued,0,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,whole,8650.0,8650.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,4225,4225.0,14.85,146.16,c,c5,rent,35000.0,source_verified,2015-12-01,issued,0,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,whole,4225.0,4225.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,10000,10000.0,11.99,222.40,c,c1,rent,42500.0,not_verified,2015-12-01,issued,0,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,whole,10000.0,10000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2016-01-01,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,20000,20000.0,10.78,432.66,b,b4,mortgage,63000.0,not_verified,2015-12-01,issued,0,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,whole,20000.0,20000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaT,0.0,2016-01-01,2015-12-01,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                    category
sub_grade                                category
home_ownership                           category
annual_inc                                float64
verification_status                      category
issue_d                            datetime64[ns]
loan_status                              category
pymnt_plan                                  int64
purpose                                  category
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


row_id                                      int64
loan_amnt                                   int64
funded_amnt                                 int64
funded_amnt_inv                           float64
int_rate                                  float64
installment                               float64
grade                                    category
sub_grade                                category
home_ownership                           category
annual_inc                                float64
verification_status                      category
issue_d                            datetime64[ns]
loan_status                              category
pymnt_plan                                  int64
purpose                                  category
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64


In [39]:
# Check columns with NaN values that are now categorical after transformations.
columns_to_check = [
    "tot_coll_amt",
    "tot_cur_bal",
    "total_rev_hi_lim",
]

# Training null report
training_null_report = (
    df_clean_training_data[columns_to_check]
    .isna()
    .agg(["sum", "mean"])
    .T
)

training_null_report["mean"] = (training_null_report["mean"] * 100).round(2)
training_null_report.columns = ["null_count_train", "null_percent_train"]

# Test null report
test_null_report = (
    df_clean_test_data[columns_to_check]
    .isna()
    .agg(["sum", "mean"])
    .T
)

test_null_report["mean"] = (test_null_report["mean"] * 100).round(2)
test_null_report.columns = ["null_count_test", "null_percent_test"]

null_comparison = training_null_report.join(test_null_report)

null_comparison


,null_count_train,null_percent_train,null_count_test,null_percent_test
tot_coll_amt,70276.0,15.07,0.0,0.0
tot_cur_bal,70276.0,15.07,0.0,0.0
total_rev_hi_lim,70276.0,15.07,0.0,0.0


In [40]:
# Drop additional columns from the clean training dataset to create the modeling datasets
df_feature_base_training_data = drop_columns_with_logging(
    dataframe=df_clean_training_data,
    columns_to_drop=excluded_columns + benchmark_columns,
    dataset_name="train_feature_base",
    log_file=project_log_file,
    errors='ignore'
    )

df_feature_base_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=excluded_columns + benchmark_columns,
    dataset_name="test_feature_base",
    log_file=project_log_file,
    errors='ignore'
    )

print(
    f"Train shape: {df_feature_base_training_data.shape} | "
    f"Test shape: {df_feature_base_test_data.shape}"
)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_feature_base_training_data.head(5))
    display(df_feature_base_test_data.head(5))

display(df_feature_base_training_data.dtypes)
display(df_feature_base_test_data.dtypes)

Train shape: (466285, 28) | Test shape: (421094, 28)


,row_id,loan_amnt,home_ownership,annual_inc,loan_status,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,5000,rent,24000.0,fully_paid,credit_card,27.65,0.0,1985-01-01,1.0,9999.0,9999.0,3.0,0.0,13648,83.7,9.0,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
1,2,2500,rent,30000.0,charged_off,car,1.00,0.0,1999-04-01,5.0,9999.0,9999.0,3.0,0.0,1687,9.4,4.0,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,60,0.0
2,3,2400,rent,12252.0,fully_paid,small_business,8.72,0.0,2001-11-01,2.0,9999.0,9999.0,2.0,0.0,2956,98.5,10.0,0.0,9999.0,0.0,NaN,NaN,NaN,0,0,0,36,10.0
3,4,10000,rent,49200.0,fully_paid,other,20.00,0.0,1996-02-01,1.0,35.0,9999.0,10.0,0.0,5598,21.0,37.0,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,36,10.0
4,5,3000,rent,80000.0,current,other,17.94,0.0,1996-01-01,0.0,38.0,9999.0,15.0,0.0,27783,53.9,38.0,0.0,9999.0,0.0,NaN,NaN,NaN,1,0,0,60,1.0


,row_id,loan_amnt,home_ownership,annual_inc,loan_status,purpose,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,has_mths_since_last_delinq,has_mths_since_last_record,has_mths_since_last_major_derog,term_months,emp_length_years
0,1,35000,mortgage,128000.0,issued,home_improvement,6.46,0.0,1990-02-01,0.0,46.0,9999.0,17.0,0.0,14277,27.4,46.0,0.0,56.0,0.0,321.0,146867.0,52200.0,1,0,1,60,10.0
1,2,8650,mortgage,100000.0,issued,credit_card,7.28,0.0,2001-07-01,0.0,9999.0,9999.0,15.0,0.0,7158,26.7,24.0,0.0,9999.0,0.0,0.0,165450.0,26800.0,0,0,0,36,0.0
2,3,4225,rent,35000.0,issued,debt_consolidation,15.22,2.0,2011-07-01,0.0,18.0,9999.0,6.0,0.0,1058,24.6,6.0,0.0,9999.0,0.0,0.0,4888.0,4300.0,1,0,0,36,5.0
3,4,10000,rent,42500.0,issued,debt_consolidation,31.04,0.0,1998-12-01,1.0,9999.0,9999.0,10.0,0.0,5812,40.9,23.0,0.0,9999.0,0.0,0.0,41166.0,14200.0,0,0,0,60,10.0
4,5,20000,mortgage,63000.0,issued,home_improvement,10.78,0.0,2000-08-01,0.0,9999.0,9999.0,6.0,0.0,7869,56.2,18.0,0.0,9999.0,0.0,0.0,189699.0,14000.0,0,0,0,60,10.0


row_id                                      int64
loan_amnt                                   int64
home_ownership                           category
annual_inc                                float64
loan_status                              category
purpose                                  category
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64
mths_since_last_record                    float64
open_acc                                  float64
pub_rec                                   float64
revol_bal                                   int64
revol_util                                float64
total_acc                                 float64
collections_12_mths_ex_med                float64
mths_since_last_major_derog               float64
acc_now_delinq                            float64


row_id                                      int64
loan_amnt                                   int64
home_ownership                           category
annual_inc                                float64
loan_status                              category
purpose                                  category
dti                                       float64
delinq_2yrs                               float64
earliest_cr_line                   datetime64[ns]
inq_last_6mths                            float64
mths_since_last_delinq                    float64
mths_since_last_record                    float64
open_acc                                  float64
pub_rec                                   float64
revol_bal                                   int64
revol_util                                float64
total_acc                                 float64
collections_12_mths_ex_med                float64
mths_since_last_major_derog               float64
acc_now_delinq                            float64


In [41]:
# Basic sanity checks on the feature base training dataset before we save them
assert "loan_status" in df_feature_base_training_data.columns
set(df_clean_training_data.columns) - set(df_feature_base_training_data.columns)


{'collection_recovery_fee',
 'funded_amnt',
 'funded_amnt_inv',
 'grade',
 'initial_list_status',
 'installment',
 'int_rate',
 'issue_d',
 'last_credit_pull_d',
 'last_pymnt_amnt',
 'last_pymnt_d',
 'next_pymnt_d',
 'out_prncp',
 'out_prncp_inv',
 'pymnt_plan',
 'recoveries',
 'sub_grade',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_int',
 'total_rec_late_fee',
 'total_rec_prncp',
 'verification_status'}

In [42]:
# Basic sanity checks on the feature base testdataset before we save them
assert "loan_status" in df_feature_base_test_data.columns
set(df_clean_test_data.columns) - set(df_feature_base_test_data.columns)

{'collection_recovery_fee',
 'funded_amnt',
 'funded_amnt_inv',
 'grade',
 'initial_list_status',
 'installment',
 'int_rate',
 'issue_d',
 'last_credit_pull_d',
 'last_pymnt_amnt',
 'last_pymnt_d',
 'next_pymnt_d',
 'out_prncp',
 'out_prncp_inv',
 'pymnt_plan',
 'recoveries',
 'sub_grade',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_int',
 'total_rec_late_fee',
 'total_rec_prncp',
 'verification_status'}

In [43]:
# Feature-base schema must match exactly (order + names) before modeling
train_columns = df_feature_base_training_data.columns.tolist()
test_columns = df_feature_base_test_data.columns.tolist()

assert train_columns == test_columns, (
    "Train/test feature_base columns are not identical.\n"
    f"Only in train: {sorted(set(train_columns) - set(test_columns))}\n"
    f"Only in test: {sorted(set(test_columns) - set(train_columns))}"
)


In [44]:
dtype_mismatches = []
for column_name in train_columns:
    if df_feature_base_training_data[column_name].dtype != df_feature_base_test_data[column_name].dtype:
        dtype_mismatches.append(
            (column_name,
             str(df_feature_base_training_data[column_name].dtype),
             str(df_feature_base_test_data[column_name].dtype))
        )

assert not dtype_mismatches, f"Dtype mismatches found: {dtype_mismatches[:20]}"


In [45]:
# Saving all datasets to interim storage as parquet files
log = get_logger(project_log_file)

log(
    f"Final shapes | "
    f"train_clean={df_clean_training_data.shape} | "
    f"test_clean={df_clean_test_data.shape} | "
    f"train_feature_base={df_feature_base_training_data.shape} | "
    f"test_feature_base={df_feature_base_test_data.shape}"
)


log(f"Saving parquet | dataset=train_clean | path={clean_training_data_file}")
df_clean_training_data.to_parquet(clean_training_data_file, index=False)

log(f"Saving parquet | dataset=test_clean | path={clean_test_data_file}")
df_clean_test_data.to_parquet(clean_test_data_file, index=False)

log(f"Saving parquet | dataset=train_feature_base | path={feature_base_training_data_file}")
df_feature_base_training_data.to_parquet(feature_base_training_data_file, index=False)

log(f"Saving parquet | dataset=test_feature_base | path={feature_base_test_data_file}")
df_feature_base_test_data.to_parquet(feature_base_test_data_file, index=False)

# Notebook Conclusion – Data Cleaning & Feature Base Construction

## Objective

The objective of this notebook was to:

1. Establish a strict **submission-time prediction boundary**
2. Eliminate structural leakage and post-origination variables
3. Normalize and standardize categorical and temporal fields
4. Ensure train/test schema alignment
5. Produce two structured outputs:
   - `clean` dataset (fully normalized, structurally consistent)
   - `feature_base` dataset (model-ready, leakage-free)

---

## What Was Completed

### 1. Structural Boundary Enforcement

- Removed identifiers and non-informative columns
- Dropped fully null variables in the 2007–2014 training set
- Excluded all post-submission and post-origination variables
- Removed geographic proxies (`addr_state`, `zip_code`) to reduce proxy discrimination risk and improve portability

Result: Feature space aligned strictly with the application submission decision point.

---

### 2. Categorical Standardization

- Lowercased all categorical text fields
- Replaced multi-word labels with underscore format (e.g., `not_verified`)
- Consolidated rare legacy categories (e.g., `none` → `other`)
- Mapped:
  - `emp_length` → ordinal numeric scale (0–10), preserving missing as unknown
  - `pymnt_plan` → binary (0/1)
  - `initial_list_status` → semantic labels (`whole`, `fractional`)
- Cast categorical variables to consistent `category` dtype
- Ensured shared category space across train and test

Result: Stable categorical schema with no drift between datasets.

---

### 3. Temporal Normalization

- Converted all month–year strings to `datetime64[ns]`
- Preserved missing values as `NaT`
- Confirmed parsing integrity (no silent coercion)

Result: Temporal features structurally valid and type-consistent.

---

### 4. Missingness Review

Identified vintage-based reporting differences:

- `tot_coll_amt`
- `tot_cur_bal`
- `total_rev_hi_lim`

Training (2007–2014): ~15% missing  
Test (2015): 0% missing  

This reflects historical reporting gaps, not pipeline error.  
Imputation strategy will be handled in modeling (train-fitted only).

---

### 5. Feature Base Dataset Creation

From the clean dataset:

- Dropped benchmark and excluded columns
- Retained only submission-time features + target
- Verified:
  - `loan_status` present
  - Identical column order (train/test)
  - Identical dtypes (train/test)

All schema validation checks passed with no mismatches.

---

## Final Outputs

Four parquet datasets were created:

- `train_clean`
- `test_clean`
- `train_feature_base`
- `test_feature_base`

Each dataset:
- Schema validated
- Dtype aligned
- Logging recorded
- Reproducible

---

## Resulting State

- No leakage variables remain in feature base.
- Train and test datasets are structurally identical.
- Category spaces are aligned.
- Datetime fields are properly typed.
- Ethical boundary (removal of geographic proxies) enforced.
- Dataset ready for EDA and modeling.

---

## Next Phase

The next notebook will:

- Define modeling cohort from `loan_status`
- Perform EDA under temporal awareness
- Evaluate feature stability and predictive leverage
- Implement imputation and encoding pipelines

The data foundation is complete.

---
